In [12]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pathlib
import shutil
import os
import requests
import time

curr_dir = pathlib.Path(".")
data_dir = curr_dir / "RTS-GMLC-master" / "RTS_Data"
wind_data_dir = curr_dir / "wind-data" / "WTK-LED" / "raw"

In [13]:
# Read in bus geodata and generation data for IEEE RTS GMLC (Reliability Test System)
df_bus = pd.read_csv(data_dir / "SourceData" / "bus.csv", index_col=[0])
df_geodata = df_bus[["lat", "lng"]]
df_gen_full = pd.read_csv(data_dir / "SourceData" / "gen.csv", index_col=[0])

In [14]:
# Get wind geodata
wind_geodata = df_geodata.loc[df_gen_full.loc[df_gen_full["Unit Type"] == "WIND", "Bus ID"]]

In [15]:
wind_geodata

,lat,lng
Bus ID,,
309,34.735758,-118.127342
317,35.378433,-117.055829
303,35.217543,-118.043550
122,32.818864,-116.324347


In [23]:
# Read in API key
with open("api-key.txt", "r") as f:
    API_KEY = f.read().strip()

# Base URL for WTK-LED Climate
BASE_URL = "https://developer.nrel.gov/api/wind-toolkit/v2/wind/wtk-led-climate-v1-0-0-download.csv"

for bus_id, row in wind_geodata.iterrows():
    print(f"Retrieving wind data for bus {bus_id}...")

    # Create output directory
    bus_wind_data_dir = wind_data_dir / f"bus{bus_id}"
    bus_wind_data_dir.mkdir(exist_ok=True, parents=True)

    lat, lon = row.lat, row.lng
    point_wkt = f"POINT({lon} {lat})"

    # WTK-LED Climate covers 2001-2020
    for year in np.arange(2001, 2021):
        print(f"Year: {year}")

        params = {
            "api_key": API_KEY,
            "wkt": point_wkt,
            "names": str(year),
            "interval": 60,  # Hourly; use 5 for 5-min if available
            "utc": "true",
            "leap_day": "true",
            "attributes": "windspeed_100m,winddirection_100m",
            "email": "charlesgulian@berkeley.edu",
            "full_name": "Charles Gulian",
            "affiliation": "UC Berkeley IEOR",
            "reason": "Research",
            "mailing_list": "false",
        }

        print("Submitting WTK-LED Climate data request to NREL...")
        response = requests.get(BASE_URL, params=params, timeout=120)

        if response.status_code == 200:
            print("Request succeeded.")
            fname = bus_wind_data_dir / f"bus{bus_id}-{year}.csv"

            # Save raw CSV
            with open(fname, "wb") as f:
                f.write(response.content)

            # Clean CSV with pandas
            df_wind = pd.read_csv(fname, skiprows=1)
            df_wind.columns = [c.strip() for c in df_wind.columns]
            df_wind["datetime"] = pd.to_datetime(
                df_wind[["Year", "Month", "Day", "Hour", "Minute"]]
            )
            df_wind = df_wind[
                ["datetime", "wind speed at 100m (m/s)", "wind direction at 100m (deg)"]
            ].set_index("datetime")
            df_wind.to_csv(fname)

            del df_wind
            del response

        else:
            print(f"Request failed with status {response.status_code}")
            print(response.text)
            break

        # Pause to avoid hitting rate limits
        time.sleep(3)

    print("Done with bus", bus_id)

Retrieving wind data for bus 309...
Year: 2001
Submitting WTK-LED Climate data request to NREL...
Request succeeded.
Year: 2002
Submitting WTK-LED Climate data request to NREL...
Request succeeded.
Year: 2003
Submitting WTK-LED Climate data request to NREL...
Request succeeded.
Year: 2004
Submitting WTK-LED Climate data request to NREL...
Request succeeded.
Year: 2005
Submitting WTK-LED Climate data request to NREL...
Request succeeded.
Year: 2006
Submitting WTK-LED Climate data request to NREL...
Request succeeded.
Year: 2007
Submitting WTK-LED Climate data request to NREL...
Request succeeded.
Year: 2008
Submitting WTK-LED Climate data request to NREL...
Request succeeded.
Year: 2009
Submitting WTK-LED Climate data request to NREL...
Request succeeded.
Year: 2010
Submitting WTK-LED Climate data request to NREL...
Request succeeded.
Year: 2011
Submitting WTK-LED Climate data request to NREL...
Request succeeded.
Year: 2012
Submitting WTK-LED Climate data request to NREL...
Request suc

In [ ]:
# Save CSV file
fname = bus_wind_data_dir / f"bus{bus_id}-{year}.csv"

# Re-open CSV with pandas and clean
df_wind = pd.read_csv(fname)

In [22]:
import requests

API_KEY = open("api-key.txt").read().strip()
url = "https://developer.nrel.gov/api/wind-toolkit/v2/wind/wtk-led-conus-download.csv"

params = {
    "api_key": API_KEY,
    "wkt": "POINT(-120.5 38.5)",
    "names": "2000",
    "interval": 60,
    "utc": "true",
    "leap_day": "true",
    "attributes": "windspeed_100m,winddirection_100m",
    "email": "charlesgulian@berkeley.edu",
    "full_name": "Charles Gulian",
    "affiliation": "UC Berkeley IEOR",
    "reason": "Research",
    "mailing_list": "false",
}

response = requests.get(url, params=params, timeout=60)
print(response.status_code)
print(response.text[:500])  # Preview first 500 chars of CSV


400
{"inputs":{"body":{},"params":{},"query":{"wkt":"POINT(-120.5 38.5)","names":"2000","interval":"60","utc":"true","leap_day":"true","attributes":"windspeed_100m,winddirection_100m","email":"charlesgulian@berkeley.edu","full_name":"Charles Gulian","affiliation":"UC Berkeley IEOR","reason":"Research","mailing_list":"false"}},"metadata":{"version":"2.0.0"},"status":400,"errors":["The required 'years' or 'names' parameter must be a single number between 2018 and 2020. The comma delimited option is on
